# Regression through the origin

Sometimes a scientific model requires the line to pass through the origin: zero input should predict zero output. Removing the intercept changes the estimator, the orthogonality conditions, and the meaning of $R^2$.


## The one-coefficient fit

The cost is now a function of a single slope,

$$
S(b) = \sum_{i=1}^{n}(y_i - b x_i)^2.
$$

Differentiate and set the derivative equal to zero:

$$
S'(b) = -2\sum_{i=1}^{n} x_i(y_i - b x_i) = 0
\Longrightarrow
b\sum x_i^2 = \sum x_i y_i
\Longrightarrow
b = \dfrac{\sum x_i y_i}{\sum x_i^2},
$$

provided $\sum x_i^2 > 0$. The second derivative $2\sum x_i^2$ is positive, so this critical point is a minimum. The residuals are orthogonal to $x$, but they need not sum to zero, because there is no intercept column forcing that condition.


## The three-point table without an intercept

The sums $\sum xy = 23$ and $\sum x^2 = 14$ are already known, so

$$
b = \dfrac{23}{14}.
$$

The predictions and residuals are

$$
\begin{aligned}
\widehat y &= \dfrac{23}{14},\ \dfrac{46}{14},\ \dfrac{69}{14}, \\
e &= 2 - \dfrac{23}{14},\ 3 - \dfrac{46}{14},\ 5 - \dfrac{69}{14}
= \dfrac{5}{14},\ -\dfrac{4}{14},\ \dfrac{1}{14}.
\end{aligned}
$$

Their sum is $2/14 \neq 0$. Their inner product with $x$ is

$$
\dfrac{5}{14} - 2\cdot\dfrac{4}{14} + 3\cdot\dfrac{1}{14} = \dfrac{5 - 8 + 3}{14} = 0,
$$

as the single normal equation requires. The cost is

$$
\mathrm{RSS}_0 = \dfrac{25 + 16 + 1}{196} = \dfrac{42}{196} = \dfrac{3}{14}.
$$

The line with an intercept had $\mathrm{RSS} = 1/6$. Since

$$
\dfrac{1}{6} - \dfrac{3}{14} = \dfrac{7 - 9}{42} = -\dfrac{1}{21} < 0,
$$

forcing the line through the origin made the fit worse. That is the usual price of a constraint the data do not support. It is the correct price when the constraint is part of the scientific model.


In [1]:
# Through-origin slope, residuals that do not sum to zero, and a larger RSS.
from fractions import Fraction
x = [Fraction(1), Fraction(2), Fraction(3)]
y = [Fraction(2), Fraction(3), Fraction(5)]
slope = sum(a * b for a, b in zip(x, y)) / sum(a * a for a in x)
errors = [yi - slope * xi for xi, yi in zip(x, y)]
rss = sum(error ** 2 for error in errors)
print("slope", slope)
print("residuals", errors)
print("sum of residuals", sum(errors))
print("sum of x times residuals", sum(xi * ei for xi, ei in zip(x, errors)))
print("RSS", rss)
assert slope == Fraction(23, 14)
assert errors == [Fraction(5, 14), Fraction(-4, 14), Fraction(1, 14)]
assert sum(errors) == Fraction(1, 7)
assert sum(xi * ei for xi, ei in zip(x, errors)) == 0
assert rss == Fraction(3, 14)
assert Fraction(1, 6) < rss


slope 23/14
residuals [Fraction(5, 14), Fraction(-2, 7), Fraction(1, 14)]
sum of residuals 1/7
sum of x times residuals 0
RSS 3/14


## One change: a line that truly passes through the origin

Take $y = 3x$ exactly, on $x = 1, 2, 3$. Then $\sum xy / \sum x^2 = 42/14 = 3$, and the residual sum of squares is zero. The intercept model also returns slope $3$ and intercept $0$. When the origin is the right constraint, the two fits agree. When it is not, the intercept model wins on residual sum of squares and the constrained model answers a different question.


In [2]:
# An exact proportional law is recovered with or without an intercept.
from fractions import Fraction
x = [Fraction(1), Fraction(2), Fraction(3)]
y = [3 * xi for xi in x]
through_origin = sum(a * b for a, b in zip(x, y)) / sum(a * a for a in x)
n = 3
sx, sy = sum(x), sum(y)
sxy = sum(a * b for a, b in zip(x, y))
sx2 = sum(a * a for a in x)
slope = (n * sxy - sx * sy) / (n * sx2 - sx ** 2)
intercept = (sy - slope * sx) / n
print("through origin", through_origin)
print("with intercept", intercept, slope)
assert through_origin == 3
assert intercept == 0 and slope == 3


through origin 3
with intercept 0 3


## Pitfall

Do not compute the ordinary $R^2 = 1 - \mathrm{RSS}/\mathrm{TSS}$ for a through-origin fit and compare it with the intercept model as though the denominators matched the same decomposition. The proof that $\mathrm{TSS} = \mathrm{SSR} + \mathrm{RSS}$ used $\sum e_i = 0$. Here $\sum e_i = 1/7 \neq 0$, so the cross term need not vanish. Judge the constraint by the residual sum of squares, or by a mean squared error that uses the right degrees of freedom: $n - 1$ after estimating one slope, not $n - 2$.


In [3]:
# The usual decomposition fails when the residuals do not sum to zero.
from fractions import Fraction
y = [Fraction(2), Fraction(3), Fraction(5)]
fitted = [Fraction(23, 14) * xi for xi in (1, 2, 3)]
mean = sum(y) / 3
tss = sum((yi - mean) ** 2 for yi in y)
ssr = sum((yh - mean) ** 2 for yh in fitted)
rss = sum((yi - yh) ** 2 for yi, yh in zip(y, fitted))
print("TSS", tss)
print("SSR + RSS", ssr + rss)
print("gap", tss - (ssr + rss))
assert rss == Fraction(3, 14)
assert tss != ssr + rss


TSS 14/3
SSR + RSS 118/21
gap -20/21


## Summary

- Through the origin, $b = \sum xy / \sum x^2$. Residuals are orthogonal to $x$ and need not add to zero.
- On the three-point table, $b = 23/14$ and $\mathrm{RSS} = 3/14$, which is worse than $1/6$.
- The intercept decomposition of the total sum of squares is not available once the residuals have a nonzero sum.
